# Generic optimizer research notebook

This notebook targets any optimizer runner module that exposes an `OPTIMIZATION` object. It runs a compact sweep, then replays one selected window with the winning parameters and emits HTML into `output/`.

Launch it from `make backtest` or `uv run python main.py` so the repo menu treats it like any other flat runner entrypoint.


In [ ]:
import importlib
from dataclasses import replace
from pprint import pprint

from backtests._shared._optimizer import run_parameter_optimization

OPTIMIZER_MODULE = "backtests.polymarket_quote_tick_pmxt_ema_optimizer"
MAX_TRIALS = 2
HOLDOUT_TOP_K = 1

optimizer_module = importlib.import_module(OPTIMIZER_MODULE)
optimization = replace(
    optimizer_module.OPTIMIZATION,
    name=f"{optimizer_module.OPTIMIZATION.name}_research",
    max_trials=min(MAX_TRIALS, optimizer_module.OPTIMIZATION.max_trials),
    holdout_top_k=min(HOLDOUT_TOP_K, optimizer_module.OPTIMIZATION.holdout_top_k),
)

summary = run_parameter_optimization(optimization)
best_params = dict(summary.selected_params)
print("Optimizer module:", OPTIMIZER_MODULE)
print("Selected params:")
pprint(best_params)
print("Leaderboard CSV:", summary.leaderboard_csv_path)
print("Summary JSON:", summary.summary_json_path)

In [ ]:
from backtests._shared._optimizer import build_optimization_window_backtest

selected_window = (
    optimization.holdout_windows[0]
    if optimization.holdout_windows
    else optimization.train_windows[-1]
)
backtest = build_optimization_window_backtest(
    config=optimization,
    window=selected_window,
    params=summary.selected_params,
    trial_id=0,
    name=f"{optimization.name}:{selected_window.name}:selected-params",
    emit_html=True,
    chart_output_path="output",
    return_summary_series=False,
)

results = backtest.run()
print(f"Replayed {len(results)} market(s) for {selected_window.name}.")
print("HTML output root:", backtest.chart_output_path)

<!-- prediction-market-backtesting:auto-embedded-html -->
Generated HTML artifacts will be embedded here after execution.
